# 01 Data Exploration & Causal Balance Diagnostics

**Uplift-DRL:** Deep Causal Learning for Promo Targeting  
Author: Samir Sandhwar (*findsamirks-commits*)

---

### 1. Setup & Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.data import generate_synthetic_uplift_data, SYNTHETIC_FEATURE_NAMES
from src.eval import qini_curve, auuc_score, uplift_decile_table
from src.targeting_bridge import TargetingBridge

print("Environment initialized successfully.")

### 2. Load or Generate Causal Benchmark Data

In [ ]:
data = generate_synthetic_uplift_data(n_samples=25000, seed=42)
X = data["X"]
T = data["T"].ravel()
Y = data["Y"].ravel()
true_ite = data["true_ite"].ravel()
fnames = data["feature_names"]

df = pd.DataFrame(X, columns=fnames)
df["treatment"] = T
df["conversion"] = Y
df["true_ite"] = true_ite

print(f"Dataset shape: {df.shape}")
df.head()

### 3. Covariate Balance Check (Standardized Mean Difference - SMD)

Verifying treatment vs. control covariate balance is critical to test for unobserved selection bias.

In [ ]:
treated = df[df["treatment"] == 1.0]
control = df[df["treatment"] == 0.0]

smd_rows = []
for col in fnames:
    m_t = treated[col].mean()
    m_c = control[col].mean()
    v_t = treated[col].var()
    v_c = control[col].var()
    pooled_sd = np.sqrt((v_t + v_c) / 2.0)
    smd = (m_t - m_c) / pooled_sd
    smd_rows.append({"feature": col, "mean_treated": m_t, "mean_control": m_c, "smd": smd})

smd_df = pd.DataFrame(smd_rows)
print("Standardized Mean Differences (|SMD| < 0.1 indicates balanced covariate distribution):")
smd_df

### 4. Overall Outcome Rates by Treatment Arm (Average Treatment Effect - ATE)

In [ ]:
ate_obs = treated["conversion"].mean() - control["conversion"].mean()
ate_true = df["true_ite"].mean()

print(f"Treated Conversion Rate : {treated['conversion'].mean()*100:.2f}%")
print(f"Control Conversion Rate : {control['conversion'].mean()*100:.2f}%")
print(f"Observed ATE (Delta)    : +{ate_obs*100:.2f} pp")
print(f"Ground Truth Mean ATE   : +{ate_true*100:.2f} pp")

### 5. Naive Decile Baseline vs. Ground-Truth CATE

In [ ]:
deciles = uplift_decile_table(Y, T, true_ite, n_bins=10)
print("Optimal Oracle Uplift Decile Table (Sorted by True ITE):")
deciles

### 6. Four-Cohort Distribution and Commercial Framing

In [ ]:
bridge = TargetingBridge(budget=50000, cost_per_treatment=20, avg_margin=120)
decisions = bridge.select(uplift_scores=true_ite)
pnl = bridge.estimate_roi(decisions)

print("Targeting Bridge Executive Summary:")
for k, v in pnl.items():
    print(f"  {k:35s}: {v}")